# Data Collection

## Setup Environment

In [2]:
import pandas as pd
import numpy as np
import hashlib
import json
import os

pd.set_option('display.max_colwidth', 120)
print('pandas version:', pd.__version__)

pandas version: 2.2.2


## Load Raw Dataset

In [3]:
RAW_SOURCE_PATH = 'KDMP.xlsx'

df_raw = pd.read_excel(RAW_SOURCE_PATH, sheet_name='Sheet1')

df_raw['create_time'] = pd.to_datetime(df_raw['create_time'])

print(f'Jumlah baris  : {len(df_raw):,}')
print(f'Jumlah kolom  : {df_raw.shape[1]}')
df_raw.head()

Jumlah baris  : 220,051
Jumlah kolom  : 8


,video_id,comment_id,parent_comment_id,level,username,nickname,comment,create_time
0,7526458489577737488,7643302845156410130,NaN,0.0,jbm123jayabetonmandiri,Christian sudartio,Beliau bilang untung nya 2 m ?,2026-05-24 10:58:57
1,7526458489577737488,7643361778294866706,7.643303e+18,1.0,sahrulmustakimm,sahrul,Makasih Mas,2026-05-24 14:47:33
2,7526458489577737488,7643368472755487495,7.643303e+18,1.0,jbm123jayabetonmandiri,Christian sudartio,Gimana untung nya 2 m..barang yg di jual ajh ga ada 1 m 😁,2026-05-24 15:13:29
3,7526458489577737488,7643374126942798599,7.643303e+18,1.0,sahrulmustakimm,sahrul,[Sticker] dibilang Makasih Mas udah 2M itu,2026-05-24 15:35:33
4,7526458489577737488,7643368138729718536,7.643303e+18,1.0,jbm123jayabetonmandiri,Christian sudartio,😳😳😳,2026-05-24 15:12:14


## Validasi Awal Data Collection

In [4]:
expected_columns = ['video_id', 'comment_id', 'parent_comment_id', 'level',
                    'username', 'nickname', 'comment', 'create_time']

checks = {
    'n_rows_raw': len(df_raw),
    'memenuhi_minimum_5000': len(df_raw) >= 5000,
    'skema_kolom_sesuai': list(df_raw.columns) == expected_columns,
    'jumlah_video_unik': int(df_raw['video_id'].nunique()),
    'rentang_waktu_mulai': str(df_raw['create_time'].min()),
    'rentang_waktu_akhir': str(df_raw['create_time'].max()),
    'tipe_data': df_raw.dtypes.astype(str).to_dict(),
}

for k, v in checks.items():
    print(f'{k:30s}: {v}')

n_rows_raw                    : 220051
memenuhi_minimum_5000         : True
skema_kolom_sesuai            : True
jumlah_video_unik             : 27
rentang_waktu_mulai           : 2025-07-13 14:06:39
rentang_waktu_akhir           : 2026-05-27 06:45:05
tipe_data                     : {'video_id': 'int64', 'comment_id': 'int64', 'parent_comment_id': 'float64', 'level': 'float64', 'username': 'object', 'nickname': 'object', 'comment': 'object', 'create_time': 'datetime64[ns]'}


In [5]:
assert checks['memenuhi_minimum_5000'], 'Data tidak memenuhi syarat minimum 5.000 baris!'
assert checks['skema_kolom_sesuai'], 'Skema kolom tidak sesuai ekspektasi!'
print('VALIDASI TAHAP DATA COLLECTION: PASSED')

VALIDASI TAHAP DATA COLLECTION: PASSED


## Simpan Raw Layer (Immutable) + Metadata Reproducibility

In [6]:
os.makedirs('../data/raw', exist_ok=True)

out_csv = '../data/raw/raw_comments_kopdes.csv'
out_parquet = '../data/raw/raw_comments_kopdes.parquet'

df_raw.to_csv(out_csv, index=False)
df_raw.to_parquet(out_parquet, index=False)

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()

metadata = {
    'source_platform': 'TikTok (video comments + replies)',
    'topic': 'Koperasi Desa Merah Putih (Kopdes)',
    'collection_method': 'TikTok comment scraper (per-video export), 27 video terkait topik Kopdes',
    'n_rows_raw': int(len(df_raw)),
    'n_unique_videos': int(df_raw['video_id'].nunique()),
    'date_range_start': str(df_raw['create_time'].min()),
    'date_range_end': str(df_raw['create_time'].max()),
    'columns': list(df_raw.columns),
    'sha256_csv': sha256(out_csv),
    'sha256_parquet': sha256(out_parquet),
}

with open('../data/raw/collection_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2, default=str)

print(json.dumps(metadata, indent=2, default=str))

{
  "source_platform": "TikTok (video comments + replies)",
  "topic": "Koperasi Desa Merah Putih (Kopdes)",
  "collection_method": "TikTok comment scraper (per-video export), 27 video terkait topik Kopdes",
  "n_rows_raw": 220051,
  "n_unique_videos": 27,
  "date_range_start": "2025-07-13 14:06:39",
  "date_range_end": "2026-05-27 06:45:05",
  "columns": [
    "video_id",
    "comment_id",
    "parent_comment_id",
    "level",
    "username",
    "nickname",
    "comment",
    "create_time"
  ],
  "sha256_csv": "bf5637a4ae81ba8411610851a47f883f337bb4233887cc423edf30d7d147e2bb",
  "sha256_parquet": "f587277b8ce852cdb65caefc1a3573b606071df24d6e0e0470a4fbc5463a3530"
}


## Kesimpulan Tahap Data Collection

| Item | Hasil |
|---|---|
| Jumlah data mentah | **220.051 baris** (>> syarat minimum 5.000) |
| Sumber | TikTok — 27 video publik tentang Koperasi Desa Merah Putih |
| Rentang waktu | 13 Jul 2025 s/d 27 Mei 2026 |
| Jenis data | Real-world, bukan dummy/sample |
| Struktur | Threaded comments (top-level + reply, ada `parent_comment_id`, `level`) |
| Status validasi | **PASSED** |
